# Playground — Step 2: Unified Guided-Generation Loop

Everything guided now runs through one loop, `_generate_guided(input_text, concepts, weights, k, steps, scorer)`.
This notebook lets you play with each path:

1. **Single concept** — `generate_with_topk_guide`, the original paper method (golden-pinned, unchanged behavior)
2. **Multi concept** — `generate_with_topk_multi_guide`, weighted sum of per-concept projections
3. **Mean frame** — `quick_generate_with_topk_subspace_guide` (currently F1.a mean-frame despite the name)
4. **Custom scorer** — the new swappable seam: write your own aggregation and pass it in

Edit prompts, concepts, weights, `k`, `steps` freely — nothing here writes to `resources/` or `tests/golden/`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [2]:
import matplotlib.pyplot as plt
import torch

from frames.representations import FrameUnembeddingRepresentation

In [3]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

2026-07-08 16:54:26.995 | INFO     | frames.models.hf.base:__init__:88 - Loaded model: hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
2026-07-08 16:54:26.998 | WARNING  | frames.models.hf.base:__init__:89 - memory cost: 5462 Mb


In [ ]:
# WordNet filter settings shared by all cells
MIN_LEMMAS = 11
MAX_TOKENS = 3


def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


PROMPTS = [
    chat("Tell me a short story.", "Once"),
    chat("Name things that make people happy.", "1."),
    chat("What men can be?", "1."),
    chat("What women can be?", "1.")
]

## 1 — Single concept (original paper method, unchanged)

In [ ]:
texts, probe = fur.quick_generate_with_topk_guide(
    PROMPTS,
    guide=["dog.n.01"],  # try ["dog.n.01", "cat.n.01"] for differential (dog - cat)
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=3,
    steps=16,
)
for t in texts:
    print(t, "\n")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.55s/it]

<|start_header_id|>user<|end_header_id|>Tell me a short story.<|start_header_id|>assistant<|end_header_id|>Once, on a mist-shrouDED island off a coast, there existed an old, 

<|start_header_id|>user<|end_header_id|>Name things that make people happy.<|start_header_id|>assistant<|end_header_id|>1. **Laugh-out Loud Moments** : Laugh out loud (LOLOLO) moments with 



## 2 — Multi concept, weighted sum

Each guide is `[synset]` or `[attract, repel]` (differential). Play with the weights —
negative weights repel. Remember the caveat: without score normalization (Step 4),
this is mathematically close to steering with one averaged frame.

In [ ]:
texts, probe = fur.quick_generate_with_topk_multi_guide(
    PROMPTS,
    guides=[["joy.n.01"], ["dog.n.01"]],
    weights=[0.7, 0.3],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=3,
    steps=16,
)
for t in texts:
    print(t, "\n")

In [ ]:
plt.figure(figsize=(8, 3))
for i, p in enumerate(probe):
    plt.plot(p.float(), label=f"prompt {i}")
plt.xlabel("token position")
plt.ylabel("cumulative combined score")
plt.legend()
plt.title("multi-guide probe")
plt.show()

## 3 — Mean-frame composition (F1.a)

Averages the guide frames into one concept (Procrustes mean) and runs vanilla
single-concept guidance. Note: named "subspace" for now; rename planned in Step 5.

In [ ]:
texts, probe = fur.quick_generate_with_topk_subspace_guide(
    PROMPTS,
    guides=[["joy.n.01"], ["dog.n.01"]],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=3,
    steps=16,
)
for t in texts:
    print(t, "\n")

## 4 — Custom scorer: the new seam

`_generate_guided` accepts any `scorer(projections, weights) -> scores` where
`projections` stacks per-concept scores on the last dim. The default is a weighted
sum; here's a **hard-min (AND)** scorer as a taste of Family 2 (proper `softmin` /
`constrained` aggregators + score normalization arrive in Step 4).

In [ ]:
def hard_min_scorer(projections: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    """AND-semantics: a candidate is only as good as its worst concept alignment."""
    return projections.min(-1).values


concept_joy = fur.get_concept("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
concept_dog = fur.get_concept("dog.n.01", MIN_LEMMAS, MAX_TOKENS)

texts, probe = fur._generate_guided(
    PROMPTS,
    concepts=[concept_joy, concept_dog],
    k=3,
    steps=16,
    scorer=hard_min_scorer,
)
for t in texts:
    print(t, "\n")

### Compare: same concepts, sum (OR-ish) vs min (AND-ish)

With the weighted sum, one strongly-aligned concept can dominate; with min, the
winning candidate must satisfy *both*. Differences are subtle at small `k` — the
candidate pool is only what the model proposes.

In [ ]:
texts_sum, _ = fur._generate_guided(
    PROMPTS, concepts=[concept_joy, concept_dog], k=3, steps=16
)
for t_min, t_sum in zip(texts, texts_sum):
    print("MIN:", t_min.split("assistant<|end_header_id|>")[-1][:140])
    print("SUM:", t_sum.split("assistant<|end_header_id|>")[-1][:140])
    print()